# IOAI — 2025 Contest Broken Bert (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/val_dataset.csv'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-broken-bert', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/val_dataset.csv' if os.path.exists('data/val_dataset.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# NeoAI 2025 — Broken BERT (고장난 BERT)

> **Northern Eurasia Olympiad in AI 2025 · Kaggle playground competition (NLP · 모델 복구)**

감성분류 BERT(`Ilseyar-kfu/broken_bert`, 3클래스)가 물에 젖어 **토큰 임베딩 일부가 0으로 손상**됐습니다
(30,522개 중 **12,208개 행이 전부 0**). 검증셋 트윗의 감성(negative/neutral/positive)을 예측합니다.

**채점**: `val_dataset.csv` 에 정답이 있어 **검증셋 macro-F1 로 로컬 채점**합니다. `submission.csv`(`id,labels`)를
만들면 Submissions 탭에서 채점됩니다. 라벨 매핑: `LABEL_0=neutral, LABEL_1=positive, LABEL_2=negative`.

⚠️ **아래 베이스라인 = 손상 모델 그대로 추론**(macro-F1 ≈ **0.29**). 0이 된 임베딩 행을 복구하면 오릅니다(모범답안 참고).

In [ ]:
import torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)
tok = AutoTokenizer.from_pretrained('Ilseyar-kfu/broken_bert')
model = AutoModelForSequenceClassification.from_pretrained('Ilseyar-kfu/broken_bert').to(dev).eval()
val = pd.read_csv('data/val_dataset.csv')
emb = model.bert.embeddings.word_embeddings.weight
n_zero = int((emb == 0).all(dim=1).sum())
print('val', len(val), '| 0 임베딩 행:', n_zero, '/', emb.shape[0])


In [ ]:
@torch.no_grad()
def predict_labels(model, texts, bs=64):
    ID2LAB = {0: 'neutral', 1: 'positive', 2: 'negative'}   # 대회 고정 매핑
    out = []
    for i in range(0, len(texts), bs):
        e = tok(texts[i:i+bs], truncation=True, padding=True, max_length=128, return_tensors='pt').to(dev)
        out += [ID2LAB[j] for j in model(**e).logits.argmax(-1).cpu().tolist()]
    return out

In [ ]:
# BASELINE: 손상 모델 그대로 검증셋 예측
from sklearn.metrics import f1_score
pred = predict_labels(model, val['text'].tolist())
print('held-out macro-F1:', round(f1_score(val['labels'], pred, average='macro'), 4))
pd.DataFrame({'id': val['id'], 'labels': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv (손상 모델 baseline — 0 임베딩 복구로 개선하세요)')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)